# Recap: Type Annotations

## Example

Here is an example with 

In [1]:
%%file example.py
from typing import reveal_type, Literal

a = 4
reveal_type(a)
print(f"a={a}")

def foo(b: int):
    reveal_type(b)
    print(f"  b={b}")

foo(a)

c: Literal[5] = 6  # FIXME
print(f"c={c}")
foo(c)

Writing example.py


At runtime, both variables `a` and `b` have type 'int':

In [2]:
%run example.py

a=4
  b=4
c=6
  b=6


Runtime type is 'int'
Runtime type is 'int'
Runtime type is 'int'


But for a static typechecker there is a difference!

In [3]:
!pyright example.py

c:\repos\pydantic_training\example.py
  c:\repos\pydantic_training\example.py:4:13 - information: Type of "a" is "Literal[4]"
  c:\repos\pydantic_training\example.py:8:17 - information: Type of "b" is "int"
  c:\repos\pydantic_training\example.py:13:17 - error: Type "Literal[6]" is not assignable to declared type "Literal[5]"
    "Literal[6]" is not assignable to type "Literal[5]" (reportAssignmentType)
1 error, 0 warnings, 2 informations 


In [4]:
from typing import reveal_type

a = 4
reveal_type(a)

Runtime type is 'int'


4

Type annotations may be inspected at runtime, too

In [5]:
%%file example2.py

from typing import get_args, get_origin, reveal_type

MyType = list[int]
print(f"MyType={MyType}")
print(f"get_origin(MyType)={get_origin(MyType)}")
print(f"get_args(MyType)={get_args(MyType)}")

my_var: MyType = [1, 2, 3]

print()
reveal_type(my_var)
print(f"my_var={my_var}")

Writing example2.py


In [6]:
%run example2.py

MyType=list[int]
get_origin(MyType)=<class 'list'>
get_args(MyType)=(<class 'int'>,)

my_var=[1, 2, 3]


Runtime type is 'list'


In [7]:
!pyright example2.py

c:\repos\pydantic_training\example2.py
  c:\repos\pydantic_training\example2.py:12:13 - information: Type of "my_var" is "list[int]"
0 errors, 0 warnings, 1 information 


# Type annotations power Pydantic

[Pydantic docs](https://docs.pydantic.dev/latest/)

In [8]:
from typing import Literal
from pydantic import BaseModel, Field

class MyModel(BaseModel):
    a: int = 6
    b: Literal[5]
    c: str = Field(..., description="This is a required string field")
    d: str = Field("just lurking", description="This is an optional string field with a default value")

print(MyModel())  # FIXME

ValidationError: 2 validation errors for MyModel
b
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing
c
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

## Loading external (JSON) data

In [9]:
%%file example.json
{
    "b": 5,
    "c": "Hello EPUG!"
}

Writing example.json


In [ ]:
from pathlib import Path

import json

with Path("example.json").open() as f:
    data = json.load(f)

print(data)
print(data["b"])
print(data["e"])  # Muhaha, you don't see it coming!

{'b': 5, 'c': 'Hello EPUG!'}
5


In [ ]:
raw_data = Path("example.json").read_bytes()  # read_text() also works
print(raw_data)
my_model = MyModel.model_validate_json(raw_data)
print(my_model)
print(my_model.b)
print(my_model.e)  # Ah, you cought me!

b'{\r\n    "b": 5,\r\n    "c": "Hello EPUG!"\r\n}\r\n'
a=6 b=5 c='Hello EPUG!' d='just lurking'
5


## Non-JSON Example

We can hand a Python dict to Pydantic, if the raw data is not in JSON format. 

In [13]:
%%file example.toml

b = 5
c = 'Hello EPUG!'

Writing example.toml


In [14]:
import tomllib

with open("example.toml", "rb") as f:
    data = tomllib.load(f)

print(data)
print(data["b"])

my_model = MyModel.model_validate(data)
print(my_model)
print(my_model.b)

{'b': 5, 'c': 'Hello EPUG!'}
5
a=6 b=5 c='Hello EPUG!' d='just lurking'
5


## Pydantic validation: pretty much anything goes

In [ ]:
from datetime import datetime
from typing import Annotated

from pydantic import StringConstraints, model_validator, field_validator, field_serializer

class EPUG_Notes(BaseModel, validate_default=True):
    created: datetime = Field(default_factory=datetime.now, description="Current timestamp")
    session_over: bool = False
    i_had: Annotated[str, StringConstraints(to_upper=True)]  = "a good time"

    @field_validator("created", mode="after")
    @classmethod
    def _validate_created(cls, value: datetime) -> datetime:
        if value.weekday() == 1:  # TODO: change discussion day so we can run this on Tuesdays
            raise ValueError("Tuesday: Time for discussion?")

        return value

    @field_serializer("session_over")
    @staticmethod
    def _serialize_session_over(value: bool) -> bool:
        if not value:
            print("Serializing `session_over` as True, even though it is False")

        return True

    @model_validator(mode="after")
    def _check_session(self):
        really_over = self.created > datetime(2025, 7, 22, 16)
        print(f"Today's session is {'over' if really_over else 'going strong'}.")
        if self.session_over != really_over:
            print(f"Overwriting `session_over` with {really_over}")
            self.session_over = really_over

        return self


session_notes = EPUG_Notes(session_over=False)
_ = Path("session_notes.json").write_text(session_notes.model_dump_json(indent=2), encoding="utf-8")

Today's session is going strong.
Serializing `session_over` as True, even though it is False


In [16]:
%pycat session_notes.json

{
  "created": "2025-07-22T13:20:37.923396",
  "session_over": true
}
